In [0]:
UC_CATALOG = "preethiworkspace_7405608993331015"
UC_SCHEMA  = "default"
VOLUME_NAME = "bank_fraud"

BASE = f"/Volumes/preethiworkspace_7405608993331015/default/bank_fraud"

paths = {
  "landing":      f"{BASE}/landing/transactions",
  "schemaLoc":    f"{BASE}/schema/transactions",
  "ckpt_bronze":  f"{BASE}/checkpoints/bronze",
  "ckpt_silver":  f"{BASE}/checkpoints/silver",
  "ckpt_gold":    f"{BASE}/checkpoints/gold",
  "reference":    f"{BASE}/reference"
}

tables = {
  "bronze": f"{UC_CATALOG}.{UC_SCHEMA}.bronze_transactions",
  "silver": f"{UC_CATALOG}.{UC_SCHEMA}.silver_transactions",
  "gold":   f"{UC_CATALOG}.{UC_SCHEMA}.gold_alerts",
  "stats":  f"{UC_CATALOG}.{UC_SCHEMA}.dim_category_stats"
}

# quick sanity check
display(dbutils.fs.ls(BASE))
paths, tables

In [0]:
import pyspark.sql.functions as F

train = (spark.read.option("header", True).option("inferSchema", True)
         .csv(f"{paths['reference']}/fraudTrain.csv"))

train = (train.withColumn("amt", F.col("amt").cast("double"))
              .withColumn("category", F.lower(F.col("category"))))

stats = (train.groupBy("category")
              .agg(F.avg("amt").alias("avg_amt_cat"),
                   F.stddev("amt").alias("std_amt_cat"),
                   F.count("*").alias("txn_cnt_cat")))

(stats.write.mode("overwrite").format("delta").saveAsTable(tables["stats"]))
